# E027 — Tri-modal fusion: MFCC + LPCC+Pitch + image

Variant of E026 where the LPCC audio branch uses the E025-winning +Pitch
augmentation (mean EER 1.94 ± 1.57%) instead of E020's +NoiseSpeed
(3.33 ± 4.14%). Everything else is identical to E026.

- **MFCC audio (E008 +All):** MFCC 13+Δ+ΔΔ+CMN, UBM-32 + MAP r=16, +Noise+Speed aug
- **LPCC+Pitch audio (E025 winner):** LPCC 13+Δ+ΔΔ+CMN (order=12), UBM-32 + MAP r=16, +Pitch aug only
- **Image (E007 +All):** PCA 50 + LogReg C=1, +flip+brightness+noise aug

Steps:
1. LOSO 3-fold (seed=67): collect OOF scores for the three systems
2. Platt calibration on OOF per system
3. Simplex grid search over (w_mfcc, w_lpcc, w_image)
4. Compare vs E023 (LPCC+image, 0.52% / 0.0104) and E026 (LPCC+NoiseSpeed tri-fusion, 0.26% / 0.0052)

In [ ]:
from pathlib import Path
import sys, copy
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import librosa
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve
from scipy.special import logsumexp
from scipy.stats import norm as scipy_norm
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    'target': '#E74C3C', 'nontarget': '#2E86AB',
    'green': '#27AE60', 'purple': '#8E44AD', 'gray': '#95A5A6',
    'mfcc': '#E67E22', 'lpcc': '#16A085', 'image': '#8E44AD',
    'fusion': '#E74C3C', 'ref': '#2E86AB', 'e026': '#F39C12',
}
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25,
})

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
SEED = 67

print(f'{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target')

## 1. Audio pipelines — MFCC (+Noise+Speed) and LPCC (+Pitch)

Both use UBM-32 + MAP r=16. MFCC uses the E008 +All augmentation (noise SNR=20dB + speed ±10%).
LPCC uses only the E025-winning +Pitch augmentation (random ±{1,2} semitones per utterance).
Val utterances scored from original WAVs only (no aug leakage).

In [ ]:
def find_wav(stem, data_dir):
    for sf in ['target_train', 'target_dev', 'non_target_train', 'non_target_dev']:
        p = data_dir / sf / (stem + '.wav')
        if p.exists():
            return p
    raise FileNotFoundError(stem)


def extract_mfcc(y, sr, n_mfcc=13):
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat   = np.vstack([mfcc, delta, delta2]).T
    feat  -= feat.mean(axis=0)
    return feat


def extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    lpcc_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        lpcc_frames.append(cep)
    feat   = np.array(lpcc_frames, dtype=np.float32)
    delta  = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat   = np.hstack([feat, delta, delta2])
    feat  -= feat.mean(axis=0)
    return feat


def aug_noise_audio(y, rng, snr_db=20.0):
    p = np.mean(y**2) + 1e-10
    return y + rng.normal(0, np.sqrt(p / 10**(snr_db/10)), len(y)).astype(y.dtype)


def aug_speed(y, rng):
    return librosa.effects.time_stretch(y, rate=rng.uniform(0.9, 1.1))


def aug_pitch(y, sr, rng):
    n_steps = float(rng.choice([-2, -1, 1, 2]))
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)


def load_and_augment_mfcc(wav_path, rng):
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    return [(y, sr), (aug_noise_audio(y, rng), sr), (aug_speed(y, rng), sr)]


def load_and_augment_lpcc_pitch(wav_path, rng):
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    # E025 +Pitch: original + pitch-shifted only (2 copies)
    return [(y, sr), (aug_pitch(y, sr, rng), sr)]


def extract_audio_batch(df, data_dir, seed, extractor, loader):
    rng = np.random.default_rng(seed)
    all_feat, all_labels = [], []
    for _, row in df.iterrows():
        for y_aug, sr in loader(find_wav(row['stem'], data_dir), rng):
            feat = extractor(y_aug, sr)
            all_feat.append(feat)
            all_labels.extend([row['label']] * len(feat))
    return np.vstack(all_feat), np.array(all_labels)


def score_utterance(wav_path, adapted, ubm, extractor):
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    feat = extractor(y, sr)
    return float((adapted.score_samples(feat) - ubm.score_samples(feat)).mean())


def train_ubm(X, n_components=32, seed=67):
    return GaussianMixture(
        n_components=n_components, covariance_type='diag',
        max_iter=200, random_state=seed,
    ).fit(X)


def map_adapt(ubm, X_target, r=16.0):
    log_prob  = ubm._estimate_log_prob(X_target)
    log_resp  = log_prob + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp      = np.exp(log_resp)
    n_k       = resp.sum(axis=0)
    mu_hat    = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha     = n_k / (n_k + r)
    adapted   = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted


print('Audio pipeline functions defined (MFCC +NoiseSpeed, LPCC +Pitch).')

## 2. Image pipeline — E007 +All

StandardScaler + PCA(50) + LogReg(C=1). Train augmented with flip + brightness[0.7,1.3] + noise σ=15; val from originals.

In [ ]:
def find_png(stem, data_dir):
    for sf in ['target_train', 'target_dev', 'non_target_train', 'non_target_dev']:
        p = data_dir / sf / (stem + '.png')
        if p.exists():
            return p
    raise FileNotFoundError(stem)


def load_image(png_path):
    img = Image.open(png_path).convert('RGB')
    if img.size != (80, 80):
        img = img.resize((80, 80), Image.BILINEAR)
    return np.array(img, dtype=np.float32).mean(axis=2).flatten()


def load_images(df, data_dir):
    return np.stack([load_image(find_png(row['stem'], data_dir)) for _, row in df.iterrows()])


def aug_flip(x):
    return x.reshape(80, 80)[:, ::-1].flatten()


def aug_brightness(x, rng):
    return np.clip(x * rng.uniform(0.7, 1.3), 0, 255)


def aug_noise_img(x, rng, sigma=15.0):
    return np.clip(x + rng.normal(0, sigma, x.shape), 0, 255)


def augment_images(X, y, seed):
    rng = np.random.default_rng(seed)
    aug_X, aug_y = [], []
    for xi, yi in zip(X, y):
        aug_X.extend([aug_flip(xi), aug_brightness(xi, rng), aug_noise_img(xi, rng)])
        aug_y.extend([yi, yi, yi])
    return np.vstack([X, np.stack(aug_X)]), np.concatenate([y, np.array(aug_y)])


print('Image pipeline functions defined.')

## 3. LOSO CV — collect OOF scores for all three systems

In [ ]:
UBM_COMPONENTS = 32
MAP_R = 16.0
N_PCA = 50
C_LOGREG = 1.0

oof_mfcc  = np.full(len(manifest), np.nan)
oof_lpcc  = np.full(len(manifest), np.nan)
oof_image = np.full(len(manifest), np.nan)

mfcc_fold_results  = []
lpcc_fold_results  = []
image_fold_results = []

print('Running LOSO CV — collecting OOF for MFCC + LPCC+Pitch + image')
print('=' * 60)

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    train_df = manifest.loc[train_idx]
    val_df   = manifest.loc[val_idx]
    y_train  = train_df['label'].to_numpy()
    y_val    = val_df['label'].to_numpy()

    print(f'\nFold {fold_id}: {len(train_df)} train, {len(val_df)} val')

    # ── MFCC audio (+Noise+Speed) ─────────────────────────────────────────
    X_m_aug, y_m_aug = extract_audio_batch(
        train_df, DATA, seed=SEED + fold_id,
        extractor=extract_mfcc, loader=load_and_augment_mfcc,
    )
    ubm_m     = train_ubm(X_m_aug[y_m_aug == 0], n_components=UBM_COMPONENTS, seed=SEED)
    adapted_m = map_adapt(ubm_m, X_m_aug[y_m_aug == 1], r=MAP_R)
    for idx, row in val_df.iterrows():
        oof_mfcc[idx] = score_utterance(find_wav(row['stem'], DATA), adapted_m, ubm_m, extract_mfcc)
    m_val = oof_mfcc[val_idx]
    eer_m, _ = compute_eer(m_val[y_val==1], m_val[y_val==0])
    dcf_m, _ = compute_min_dcf(m_val[y_val==1], m_val[y_val==0])
    mfcc_fold_results.append({'fold': fold_id, 'eer': eer_m, 'min_dcf': dcf_m})
    print(f'  MFCC         EER={eer_m*100:.2f}%  min-DCF={dcf_m:.4f}')

    # ── LPCC audio (+Pitch) ───────────────────────────────────────────────
    X_l_aug, y_l_aug = extract_audio_batch(
        train_df, DATA, seed=SEED + fold_id,
        extractor=extract_lpcc, loader=load_and_augment_lpcc_pitch,
    )
    ubm_l     = train_ubm(X_l_aug[y_l_aug == 0], n_components=UBM_COMPONENTS, seed=SEED)
    adapted_l = map_adapt(ubm_l, X_l_aug[y_l_aug == 1], r=MAP_R)
    for idx, row in val_df.iterrows():
        oof_lpcc[idx] = score_utterance(find_wav(row['stem'], DATA), adapted_l, ubm_l, extract_lpcc)
    l_val = oof_lpcc[val_idx]
    eer_l, _ = compute_eer(l_val[y_val==1], l_val[y_val==0])
    dcf_l, _ = compute_min_dcf(l_val[y_val==1], l_val[y_val==0])
    lpcc_fold_results.append({'fold': fold_id, 'eer': eer_l, 'min_dcf': dcf_l})
    print(f'  LPCC+Pitch   EER={eer_l*100:.2f}%  min-DCF={dcf_l:.4f}')

    # ── Image ─────────────────────────────────────────────────────────────
    X_train_orig = load_images(train_df, DATA)
    X_val_orig   = load_images(val_df,   DATA)
    X_train_aug, y_train_aug = augment_images(X_train_orig, y_train, seed=SEED + fold_id)

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_train_aug)
    X_vl_s = scaler.transform(X_val_orig)

    pca = PCA(n_components=N_PCA, random_state=SEED)
    X_tr_p = pca.fit_transform(X_tr_s)
    X_vl_p = pca.transform(X_vl_s)

    clf = LogisticRegression(C=C_LOGREG, max_iter=1000, random_state=SEED)
    clf.fit(X_tr_p, y_train_aug)
    oof_image[val_idx] = clf.decision_function(X_vl_p)

    i_val = oof_image[val_idx]
    eer_i, _ = compute_eer(i_val[y_val==1], i_val[y_val==0])
    dcf_i, _ = compute_min_dcf(i_val[y_val==1], i_val[y_val==0])
    image_fold_results.append({'fold': fold_id, 'eer': eer_i, 'min_dcf': dcf_i})
    print(f'  Image        EER={eer_i*100:.2f}%  min-DCF={dcf_i:.4f}')

print('\n' + '=' * 60)
print('OOF collection complete.')

mfcc_eers  = [r['eer']*100 for r in mfcc_fold_results]
lpcc_eers  = [r['eer']*100 for r in lpcc_fold_results]
image_eers = [r['eer']*100 for r in image_fold_results]
print(f'\nMFCC per-fold EER:       {mfcc_eers[0]:.2f}, {mfcc_eers[1]:.2f}, {mfcc_eers[2]:.2f}  '
      f'mean={np.mean(mfcc_eers):.2f}±{np.std(mfcc_eers):.2f}%')
print(f'LPCC+Pitch per-fold EER: {lpcc_eers[0]:.2f}, {lpcc_eers[1]:.2f}, {lpcc_eers[2]:.2f}  '
      f'mean={np.mean(lpcc_eers):.2f}±{np.std(lpcc_eers):.2f}%')
print(f'Image per-fold EER:      {image_eers[0]:.2f}, {image_eers[1]:.2f}, {image_eers[2]:.2f}  '
      f'mean={np.mean(image_eers):.2f}±{np.std(image_eers):.2f}%')

## 4. Platt calibration

In [ ]:
def platt_calibrate(scores, labels):
    cal = LogisticRegression(C=1e6, max_iter=1000, class_weight='balanced', random_state=SEED)
    cal.fit(scores.reshape(-1, 1), labels)
    return cal.decision_function(scores.reshape(-1, 1))


oof_mfcc_cal  = platt_calibrate(oof_mfcc,  y_all)
oof_lpcc_cal  = platt_calibrate(oof_lpcc,  y_all)
oof_image_cal = platt_calibrate(oof_image, y_all)

eer_m_cal, _ = compute_eer(oof_mfcc_cal[y_all==1],  oof_mfcc_cal[y_all==0])
eer_l_cal, _ = compute_eer(oof_lpcc_cal[y_all==1],  oof_lpcc_cal[y_all==0])
eer_i_cal, _ = compute_eer(oof_image_cal[y_all==1], oof_image_cal[y_all==0])
dcf_m_cal, _ = compute_min_dcf(oof_mfcc_cal[y_all==1],  oof_mfcc_cal[y_all==0])
dcf_l_cal, _ = compute_min_dcf(oof_lpcc_cal[y_all==1],  oof_lpcc_cal[y_all==0])
dcf_i_cal, _ = compute_min_dcf(oof_image_cal[y_all==1], oof_image_cal[y_all==0])

print('OOF EER (calibrated):')
print(f'  MFCC:        {eer_m_cal*100:.2f}%  min-DCF={dcf_m_cal:.4f}')
print(f'  LPCC+Pitch:  {eer_l_cal*100:.2f}%  min-DCF={dcf_l_cal:.4f}')
print(f'  Image:       {eer_i_cal*100:.2f}%  min-DCF={dcf_i_cal:.4f}')

cor_ml = np.corrcoef(oof_mfcc_cal,  oof_lpcc_cal)[0,1]
cor_mi = np.corrcoef(oof_mfcc_cal,  oof_image_cal)[0,1]
cor_li = np.corrcoef(oof_lpcc_cal,  oof_image_cal)[0,1]
print('\nPairwise Pearson correlations (calibrated):')
print(f'  MFCC–LPCC:  {cor_ml:.3f}')
print(f'  MFCC–Image: {cor_mi:.3f}')
print(f'  LPCC–Image: {cor_li:.3f}')

## 5. Simplex grid search over (w_mfcc, w_lpcc, w_image)

In [ ]:
weights_grid = np.linspace(0, 1, 51)
n = len(weights_grid)
eer_grid = np.full((n, n), np.nan)
dcf_grid = np.full((n, n), np.nan)

best_eer = np.inf; best_w = None
best_dcf = np.inf; best_w_dcf = None

for i, w_m in enumerate(weights_grid):
    for j, w_l in enumerate(weights_grid):
        if w_m + w_l > 1.0 + 1e-9:
            continue
        w_i = 1.0 - w_m - w_l
        fused = w_m * oof_mfcc_cal + w_l * oof_lpcc_cal + w_i * oof_image_cal
        eer, _ = compute_eer(fused[y_all==1], fused[y_all==0])
        dcf, _ = compute_min_dcf(fused[y_all==1], fused[y_all==0])
        eer_grid[i, j] = eer
        dcf_grid[i, j] = dcf
        if eer < best_eer:
            best_eer = eer; best_w = (w_m, w_l, w_i)
        if dcf < best_dcf:
            best_dcf = dcf; best_w_dcf = (w_m, w_l, w_i)

w_m_best, w_l_best, w_i_best = best_w
w_m_dcf,  w_l_dcf,  w_i_dcf  = best_w_dcf
scores_grid     = w_m_best * oof_mfcc_cal + w_l_best * oof_lpcc_cal + w_i_best * oof_image_cal
scores_grid_dcf = w_m_dcf  * oof_mfcc_cal + w_l_dcf  * oof_lpcc_cal + w_i_dcf  * oof_image_cal
dcf_at_eer_best, _ = compute_min_dcf(scores_grid[y_all==1], scores_grid[y_all==0])
eer_at_dcf_best, _ = compute_eer(scores_grid_dcf[y_all==1], scores_grid_dcf[y_all==0])

print(f'Best grid (EER):  w_mfcc={w_m_best:.2f}, w_lpcc={w_l_best:.2f}, w_image={w_i_best:.2f}')
print(f'                  OOF EER={best_eer*100:.2f}%, min-DCF={dcf_at_eer_best:.4f}')
print(f'Best grid (DCF):  w_mfcc={w_m_dcf:.2f}, w_lpcc={w_l_dcf:.2f}, w_image={w_i_dcf:.2f}')
print(f'                  OOF EER={eer_at_dcf_best*100:.2f}%, min-DCF={best_dcf:.4f}')

## 6. LogReg fusion on 3 calibrated OOF columns (reference)

In [ ]:
X_fuse = np.column_stack([oof_mfcc_cal, oof_lpcc_cal, oof_image_cal])
fuser = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
fuser.fit(X_fuse, y_all)
scores_logreg = fuser.decision_function(X_fuse)
eer_lr, _ = compute_eer(scores_logreg[y_all==1], scores_logreg[y_all==0])
dcf_lr, _ = compute_min_dcf(scores_logreg[y_all==1], scores_logreg[y_all==0])

lr_coef = fuser.coef_.ravel()
lr_norm = lr_coef / (np.abs(lr_coef).sum() + 1e-12)
print(f'LogReg fusion: OOF EER={eer_lr*100:.2f}%  min-DCF={dcf_lr:.4f}')
print(f'LogReg coefficients (raw):      mfcc={lr_coef[0]:+.3f}  lpcc={lr_coef[1]:+.3f}  image={lr_coef[2]:+.3f}')
print(f'LogReg intercept:               {fuser.intercept_[0]:+.3f}')
print(f'LogReg |coef| normalised (L1):  mfcc={lr_norm[0]:+.3f}  lpcc={lr_norm[1]:+.3f}  image={lr_norm[2]:+.3f}')

## 7. Results table — vs E023 and E026

In [ ]:
rows = [
    ('MFCC audio (E008 +All)',                    eer_m_cal*100, dcf_m_cal),
    ('LPCC+Pitch audio (E025 winner)',            eer_l_cal*100, dcf_l_cal),
    ('Image (E007 +All)',                         eer_i_cal*100, dcf_i_cal),
    ('E023 LPCC+image grid (w=0.36)',             0.52,          0.0104),
    ('E026 MFCC+LPCC+image grid (0.06/0.52/0.42)',0.26,          0.0052),
    (f'E027 MFCC+LPCC+Pitch+image grid EER ({w_m_best:.2f}/{w_l_best:.2f}/{w_i_best:.2f})',
        best_eer*100, dcf_at_eer_best),
    (f'E027 MFCC+LPCC+Pitch+image grid DCF ({w_m_dcf:.2f}/{w_l_dcf:.2f}/{w_i_dcf:.2f})',
        eer_at_dcf_best*100, best_dcf),
    ('E027 MFCC+LPCC+Pitch+image LogReg',         eer_lr*100,    dcf_lr),
]

print(f"{'System':<62} {'OOF EER [%]':>12} {'min-DCF':>10}")
print('-' * 88)
for name, eer_pct, dcf in rows:
    print(f'{name:<62} {eer_pct:>12.2f} {dcf:>10.4f}')
print('-' * 88)

print('\nDelta vs E026 (0.26% / 0.0052):')
print(f'  Grid (EER-opt):  {best_eer*100 - 0.26:+.2f}pp EER, {dcf_at_eer_best - 0.0052:+.4f} min-DCF')
print(f'  LogReg fusion:   {eer_lr*100 - 0.26:+.2f}pp EER, {dcf_lr - 0.0052:+.4f} min-DCF')
print('\nDelta vs E023 (0.52% / 0.0104):')
print(f'  Grid (EER-opt):  {best_eer*100 - 0.52:+.2f}pp EER, {dcf_at_eer_best - 0.0104:+.4f} min-DCF')
print(f'  LogReg fusion:   {eer_lr*100 - 0.52:+.2f}pp EER, {dcf_lr - 0.0104:+.4f} min-DCF')

## 8. Grid heatmap + 1D slice

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
disp = eer_grid.copy() * 100
im = ax.imshow(disp, origin='lower', extent=[0, 1, 0, 1],
               aspect='auto', cmap='viridis_r', vmin=0,
               vmax=max(10, np.nanpercentile(disp, 95)))
ax.set_xlabel('w_lpcc')
ax.set_ylabel('w_mfcc')
ax.set_title('OOF EER [%] on simplex\n(white = infeasible w_m+w_l>1)')
cb = plt.colorbar(im, ax=ax); cb.set_label('OOF EER [%]')
ax.scatter([w_l_best], [w_m_best], color='red', s=140, marker='*', zorder=5,
           label=f'E027 best ({w_m_best:.2f}, {w_l_best:.2f}, {w_i_best:.2f})')
ax.scatter([0.52], [0.06], color='white', s=100, marker='x', zorder=5,
           label='E026 ref (0.06, 0.52, 0.42)')
ax.scatter([0.36], [0.00], color='yellow', s=90, marker='o', zorder=5,
           label='E023 ref (0, 0.36, 0.64)')
ax.legend(fontsize=8)

ax = axes[1]
w_m_scan = np.linspace(0, 1 - w_i_best, 101)
eer_scan = []
for wm in w_m_scan:
    wl = 1 - wm - w_i_best
    if wl < 0:
        eer_scan.append(np.nan); continue
    s = wm * oof_mfcc_cal + wl * oof_lpcc_cal + w_i_best * oof_image_cal
    e, _ = compute_eer(s[y_all==1], s[y_all==0])
    eer_scan.append(e*100)
ax.plot(w_m_scan, eer_scan, lw=2, color=COLORS['fusion'],
        label=f'w_image fixed at {w_i_best:.2f} (best)')
ax.axvline(w_m_best, color=COLORS['green'], ls='--', lw=2,
           label=f'best w_mfcc={w_m_best:.2f}')
ax.axhline(0.26, color=COLORS['e026'], ls=':', lw=1.5, label='E026 (0.26%)')
ax.axhline(0.52, color=COLORS['ref'], ls=':', lw=1.5, label='E023 (0.52%)')
ax.set_xlabel('w_mfcc')
ax.set_ylabel('OOF EER [%]')
ax.set_title('1D slice — vary w_mfcc, w_image fixed, w_lpcc = 1 − w_mfcc − w_image')
ax.legend(fontsize=8)

plt.suptitle('E027 — Tri-modal simplex grid search (LPCC+Pitch)', fontsize=12)
plt.tight_layout()
plt.show()

## 9. Score distribution at best fusion

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
bins = np.linspace(scores_grid.min(), scores_grid.max(), 40)
ax.hist(scores_grid[y_all==0], bins=bins, alpha=0.75, color=COLORS['nontarget'],
        label=f'non-target  (n={int((y_all==0).sum())})')
ax.hist(scores_grid[y_all==1], bins=bins, alpha=0.75, color=COLORS['target'],
        label=f'target  (n={int((y_all==1).sum())})')
ax.set_xlabel('Fused score (log-odds, calibrated)')
ax.set_ylabel('count')
ax.set_title(f'E027 best grid fusion score distribution — OOF EER={best_eer*100:.2f}%')
ax.legend()
plt.tight_layout(); plt.show()

## 10. DET curves — E023, E026, E027

In [ ]:
# Reconstruct an E026-style fusion from the current calibrated scores using E026 grid weights.
# Note: this is E026's simplex *applied to E027's LPCC+Pitch OOF* — only useful as a weak proxy.
# For the actual E026 reference we use the published numbers (0.26% / 0.0052).

ticks = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]
tick_pos    = [scipy_norm.ppf(t) for t in ticks]
tick_labels = [f'{int(t*100)}' for t in ticks]

fig, ax = plt.subplots(figsize=(8, 7))

# Per-modality calibrated OOF (thin dashed)
det_systems = [
    ('MFCC audio',                  oof_mfcc_cal,   COLORS['mfcc'],  '--', 1.2),
    ('LPCC+Pitch audio',            oof_lpcc_cal,   COLORS['lpcc'],  '--', 1.2),
    ('Image',                       oof_image_cal,  COLORS['image'], '--', 1.2),
    ('E023 LPCC+image w=0.36',      0.36*oof_lpcc_cal + 0.64*oof_image_cal,
                                                    COLORS['ref'],   '-',  1.5),
    (f'E027 grid ({w_m_best:.2f}/{w_l_best:.2f}/{w_i_best:.2f})',
                                    scores_grid,    COLORS['fusion'],'-',  2.5),
    ('E027 LogReg fusion',          scores_logreg,  COLORS['green'], '-',  1.8),
]
for name, scores, color, ls, lw in det_systems:
    fpr, tpr, _ = roc_curve(y_all, scores)
    far_c = np.clip(fpr, 1e-4, 1-1e-4)
    frr_c = np.clip(1-tpr, 1e-4, 1-1e-4)
    eer_c, _ = compute_eer(scores[y_all==1], scores[y_all==0])
    ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c),
            color=color, lw=lw, ls=ls,
            label=f'{name}  EER={eer_c*100:.2f}%')

ax.plot(tick_pos, tick_pos, 'k--', lw=1)
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels)
ax.set_xlabel('FAR [%]'); ax.set_ylabel('FRR [%]')
ax.set_title('DET curves — E027 (LPCC+Pitch tri-fusion) vs baselines (OOF)')
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout(); plt.show()

## 11. Weight summary

In [ ]:
print('E027 — tri-modal fusion (LPCC+Pitch) weight summary')
print('=' * 60)
print('Grid search (EER-optimal):')
print(f'  w_mfcc  = {w_m_best:.2f}')
print(f'  w_lpcc  = {w_l_best:.2f}')
print(f'  w_image = {w_i_best:.2f}')
print(f'  → OOF EER = {best_eer*100:.2f}%, min-DCF = {dcf_at_eer_best:.4f}')
print()
print('Grid search (min-DCF-optimal):')
print(f'  w_mfcc  = {w_m_dcf:.2f}')
print(f'  w_lpcc  = {w_l_dcf:.2f}')
print(f'  w_image = {w_i_dcf:.2f}')
print(f'  → OOF EER = {eer_at_dcf_best*100:.2f}%, min-DCF = {best_dcf:.4f}')
print()
print('LogReg fusion coefficients:')
print(f'  mfcc={lr_coef[0]:+.3f}  lpcc={lr_coef[1]:+.3f}  image={lr_coef[2]:+.3f}  bias={fuser.intercept_[0]:+.3f}')
print(f'  → OOF EER = {eer_lr*100:.2f}%, min-DCF = {dcf_lr:.4f}')
print()
print('Reference fusions:')
print(f'  E023 LPCC+image (w=0.36):                EER=0.52%, min-DCF=0.0104')
print(f'  E026 MFCC+LPCC+image (0.06/0.52/0.42):   EER=0.26%, min-DCF=0.0052')